## Setup
Before running this notebook, please follow the instructions in the `README.md` file from the repo root to set up your environment.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

ENV_KEYS = [
    "OPENAI_API_KEY",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "MODAL_TOKEN_ID",
    "MODAL_TOKEN_SECRET",
]

missing_keys = [k for k in ENV_KEYS if not os.getenv(k)]
if missing_keys:
    print(
        "Env vars not set (needed for some sections): "
        + ", ".join(missing_keys)
    )


In [2]:
# Basic Imports
import json
from pathlib import Path
import pandas as pd
import random
import numpy as np
import json
import time
import re
import litellm

## Generate a Mini-Query Set
We use TPC-DS as our database and generate a small set of queries at scale factor 3.

In [ ]:
from dbplanbench import generate_queries

query_generation_results = generate_queries(
    dataset='tpcds',
    complexity_distribution={
        5: 2,
        6: 1,
    },
    n_queries=3,
    run_dir="test_query_generation",
    scale_factor=3,
)

## Optimize Queries

In [1]:
# The query that admits good optimization in tpcds!

good_query = """SELECT
  d.d_year AS sales_year,
  s.channel,
  ca.ca_state,
  i.i_category,
  SUM(s.sales_amt) AS total_sales,
  COALESCE(SUM(s.return_amt), 0) AS total_returns,
  SUM(s.sales_amt) - COALESCE(SUM(s.return_amt), 0) AS net_sales
FROM
(
  SELECT
    'store'::text AS channel,
    ss.ss_sold_date_sk AS sold_date_sk,
    ss.ss_item_sk AS item_sk,
    ss.ss_addr_sk AS addr_sk,
    ss.ss_net_paid AS sales_amt,
    sr_ret.return_amt AS return_amt
  FROM store_sales ss
  LEFT JOIN (
    SELECT sr_item_sk, sr_ticket_number, SUM(sr_return_amt) AS return_amt
    FROM store_returns
    GROUP BY sr_item_sk, sr_ticket_number
  ) sr_ret
    ON sr_ret.sr_item_sk = ss.ss_item_sk
   AND sr_ret.sr_ticket_number = ss.ss_ticket_number

  UNION ALL

  SELECT
    'web'::text AS channel,
    ws.ws_sold_date_sk AS sold_date_sk,
    ws.ws_item_sk AS item_sk,
    ws.ws_bill_addr_sk AS addr_sk,
    ws.ws_net_paid AS sales_amt,
    wr_ret.return_amt AS return_amt
  FROM web_sales ws
  LEFT JOIN (
    SELECT wr_item_sk, wr_order_number, SUM(wr_return_amt) AS return_amt
    FROM web_returns
    GROUP BY wr_item_sk, wr_order_number
  ) wr_ret
    ON wr_ret.wr_item_sk = ws.ws_item_sk
   AND wr_ret.wr_order_number = ws.ws_order_number

  UNION ALL

  SELECT
    'catalog'::text AS channel,
    cs.cs_sold_date_sk AS sold_date_sk,
    cs.cs_item_sk AS item_sk,
    cs.cs_bill_addr_sk AS addr_sk,
    cs.cs_net_paid AS sales_amt,
    cr_ret.return_amt AS return_amt
  FROM catalog_sales cs
  LEFT JOIN (
    SELECT cr_item_sk, cr_order_number, SUM(cr_return_amount) AS return_amt
    FROM catalog_returns
    GROUP BY cr_item_sk, cr_order_number
  ) cr_ret
    ON cr_ret.cr_item_sk = cs.cs_item_sk
   AND cr_ret.cr_order_number = cs.cs_order_number
) s
JOIN item i ON i.i_item_sk = s.item_sk
JOIN customer_address ca ON ca.ca_address_sk = s.addr_sk
JOIN date_dim d ON d.d_date_sk = s.sold_date_sk
WHERE d.d_year = 2001
  AND (
    SELECT COUNT(*)
    FROM warehouse w
    WHERE w.w_state = ca.ca_state
  ) >= 2
GROUP BY
  d.d_year,
  s.channel,
  ca.ca_state,
  i.i_category
HAVING
  SUM(s.sales_amt) > CAST(100000 AS decimal(12,2))
  AND COALESCE(SUM(s.return_amt), 0) < SUM(s.sales_amt) * 0.1
ORDER BY
  d.d_year,
  s.channel,
  ca.ca_state,
  i.i_category
"""

queries = [good_query]

In [ ]:
from dbplanbench import optimize_queries

results = optimize_queries(
    queries=queries,
    run_dir="test_optimization_run",
    dataset="tpcds",
    n_steps=1,
    n_samples_per_step=2,
    n_runs=100,
    top_k_patches=1,
    scale_factor=3,
    max_workers=1, # single query, so no need for parallelism here
    max_eval_workers=50, # let's parallelize evaluation well
    get_full_metrics=True, # let's return full metrics
)

## Transfer Plans to SF 6

In [ ]:
from dbplanbench import scale_optimizations

scale_result = scale_optimizations(
    queries=queries,
    plans=results.optimization_outcome,
    source_scale_factor=3,
    target_scale_factor=6,
    dataset="tpcds",
    run_dir="test_scale_optimizations",
    modal_resources={"memory": (8 * 1024, 8 * 1024)},
)

print(f"Run dir: {scale_result.run_dir}")
print(f"Summary: {json.dumps(scale_result.summary, indent=2)}")
for i, (q, fail) in enumerate(zip(queries, scale_result.transfer_failures)):
    status = "OK" if fail is None else f"FAILED: {fail}"
    print(f"  Query {i}: {status}")

## Benchmark at Small Scale (SF3)
For each query, benchmark both the base engine plan (empty patch) and the optimized plan (with patch). This confirms the optimization improved performance at the original scale.

In [ ]:
from dbplanbench import benchmark_plans
from dbplanbench_types import PatchedPlan

# Build PatchedPlans with multi-patch [[], optimization_patch] so benchmark_plans
# evaluates both the base engine plan and the optimized plan for each query.
small_scale_plans = []
for pp in results.optimization_outcome:
    # First patch [] = base plan, second patch = optimization
    best_patch = pp.patch[0] if pp.patch[0] is not None else []
    small_scale_plans.append(PatchedPlan(
        base_plan=pp.base_plan,
        patch=[[], best_patch],
    ))

small_benchmark = benchmark_plans(
    small_scale_plans,
    dataset="tpcds",
    scale_factor=3,
    n_runs=100,
    max_workers=1,
    max_eval_workers=50,
)

print("Small-scale benchmark results:")
for i, per_query in enumerate(small_benchmark.results):
    base_stats = per_query[0] if len(per_query) > 0 else {}
    opt_stats = per_query[1] if len(per_query) > 1 else {}
    base_min = base_stats.get("benchmark_stats", {}).get("execution_time", {}).get("min", "N/A")
    opt_min = opt_stats.get("benchmark_stats", {}).get("execution_time", {}).get("min", "N/A")
    print(f"  Query {i}: base={base_min}, optimized={opt_min}")

## Benchmark at Large Scale (SF6)
For each query, benchmark both the transferred base plan (empty patch) and the transferred optimized plan (with transferred patch). This verifies the optimization benefit survived the scale transfer.

In [ ]:
# Build PatchedPlans for SF6: transferred base + [[], transferred_patch]
large_scale_plans = []
for pp in scale_result.scaled_plans:
    best_patch = pp.patch[0] if pp.patch[0] is not None else []
    large_scale_plans.append(PatchedPlan(
        base_plan=pp.base_plan,
        patch=[[], best_patch],
    ))

large_benchmark = benchmark_plans(
    large_scale_plans,
    dataset="tpcds",
    scale_factor=6,
    n_runs=100,
    memory=(8 * 1024, 8 * 1024),
    max_workers=1,
    max_eval_workers=50,
)

print("Large-scale benchmark results:")
for i, per_query in enumerate(large_benchmark.results):
    base_stats = per_query[0] if len(per_query) > 0 else {}
    opt_stats = per_query[1] if len(per_query) > 1 else {}
    base_min = base_stats.get("benchmark_stats", {}).get("execution_time", {}).get("min", "N/A")
    opt_min = opt_stats.get("benchmark_stats", {}).get("execution_time", {}).get("min", "N/A")
    print(f"  Query {i}: base={base_min}, optimized={opt_min}")

## Results Comparison
Compare optimization speedups across both scale factors to verify that the transferred optimizations preserve their benefits at the larger scale.

In [ ]:
def extract_min_elapsed(per_query_results, idx):
    """Extract mean elapsed time from benchmark results at a given patch index."""
    if idx >= len(per_query_results):
        return None
    stats = per_query_results[idx]
    if not stats or "error" in stats:
        return None
    return stats.get("benchmark_stats", {}).get("execution_time", {}).get("min")

rows = []
for i in range(len(queries)):
    small_base = extract_min_elapsed(small_benchmark.results[i], 0)
    small_opt = extract_min_elapsed(small_benchmark.results[i], 1)
    large_base = extract_min_elapsed(large_benchmark.results[i], 0)
    large_opt = extract_min_elapsed(large_benchmark.results[i], 1)

    def speedup(base, opt):
        if base and opt and opt > 0:
            return f"{base / opt:.2f}x"
        return "N/A"

    rows.append({
        "Query": i,
        "SF3 Base (s)": f"{small_base:.1f}" if small_base else "ERR",
        "SF3 Opt (s)": f"{small_opt:.1f}" if small_opt else "ERR",
        "SF3 Speedup": speedup(small_base, small_opt),
        "SF6 Base (s)": f"{large_base:.1f}" if large_base else "ERR",
        "SF6 Opt (s)": f"{large_opt:.1f}" if large_opt else "ERR",
        "SF6 Speedup": speedup(large_base, large_opt),
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))